# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [3]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [4]:
# 1
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])

# Как на уроке: строки — города, столбцы — месяцы.
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])
monthly_revenue = table.sum(axis=0)
september, october = monthly_revenue
change = (october - september) / september * 100

print(f"Чистых заказов: {clean.sum()}")
print(f"Сентябрь: {september:,.0f} ₸")
print(f"Октябрь: {october:,.0f} ₸")
print(f"Изменение: {change:.2f}%")
print("В пункте 5 ДЗ 2 падение было 22,7%, а после удаления дублей и TEST выручка упала только на 2,34%.")

Чистых заказов: 4739
Сентябрь: 35,243,410 ₸
Октябрь: 34,419,520 ₸
Изменение: -2.34%
В пункте 5 ДЗ 2 падение было 22,7%, а после удаления дублей и TEST выручка упала только на 2,34%.


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [5]:
# 2
categories = np.unique(cat)

# Таблица по образцу из урока: категории вместо городов.
category_revenue = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months]
                            for c in categories])

category_total = category_revenue.sum(axis=1)
category_change = (category_revenue[:, 1] - category_revenue[:, 0]) / category_revenue[:, 0] * 100

print(f"{'Категория':<12} {'Сентябрь, ₸':>14} {'Октябрь, ₸':>14} {'Изменение, %':>13}")
for category, revenue, rate in zip(categories, category_revenue, category_change):
    print(f"{category:<12} {revenue[0]:>14,.0f} {revenue[1]:>14,.0f} {rate:>13.2f}")

top = category_total.argmax()
# Самое большое падение соответствует максимуму изменения с обратным знаком.
worst = (-category_change).argmax()
print(f"За два месяца больше всего принесла категория «{categories[top]}» — {category_total[top]:,.0f} ₸, "
      f"а сильнее всего упала категория «{categories[worst]}» — на {-category_change[worst]:.2f}%.")

Категория       Сентябрь, ₸     Октябрь, ₸  Изменение, %
Дом               4,997,730      5,125,790          2.56
Книги             1,065,300      1,140,910          7.10
Красота           2,610,950      2,678,990          2.61
Одежда            9,167,100      8,505,730         -7.21
Электроника      17,402,330     16,968,100         -2.50
За два месяца больше всего принесла категория «Электроника» — 34,370,430 ₸, а сильнее всего упала категория «Одежда» — на 7.21%.


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [6]:
# 3
counts = np.array([[(clean & (cat == c) & (month == m)).sum() for m in months]
                   for c in categories])

average_check = category_revenue / counts
october_check = average_check[:, 1]

print(f"{'Категория':<12} {'Заказы в сен.':>13} {'Заказы в окт.':>13} {'Чек в окт., ₸':>14}")
for category, order_counts, check in zip(categories, counts, october_check):
    print(f"{category:<12} {order_counts[0]:>13} {order_counts[1]:>13} {check:>14,.2f}")

top = october_check.argmax()
print(f"Самый высокий средний чек в октябре у категории «{categories[top]}» — {october_check[top]:,.2f} ₸.")

Категория    Заказы в сен. Заказы в окт.  Чек в окт., ₸
Дом                    476           476      10,768.47
Книги                  265           299       3,815.75
Красота                422           435       6,158.60
Одежда                 678           652      13,045.60
Электроника            531           505      33,600.20
Самый высокий средний чек в октябре у категории «Электроника» — 33,600.20 ₸.


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [7]:
# 4
city_share = table / table.sum(axis=0)
column_sums = city_share.sum(axis=0)

print(f"{'Город':<12} {'Сентябрь':>10} {'Октябрь':>10}")
for current_city, shares in zip(cities, city_share):
    print(f"{current_city:<12} {shares[0]:>10.2%} {shares[1]:>10.2%}")

print("Суммы долей по месяцам:", column_sums)
print("Сумма равна 1 в сентябре:", round(column_sums[0], 6) == 1)
print("Сумма равна 1 в октябре:", round(column_sums[1], 6) == 1)

shymkent_share = city_share[cities == "Шымкент"][0]
share_change = (shymkent_share[1] - shymkent_share[0]) * 100
print(f"Доля Шымкента снизилась с {shymkent_share[0]:.2%} до {shymkent_share[1]:.2%}, "
      f"изменение составило {share_change:.2f} п. п.")

Город          Сентябрь    Октябрь
Актобе            7.31%      7.38%
Алматы           41.15%     42.54%
Астана           25.63%     25.71%
Атырау            2.62%      2.82%
Караганда        10.35%      9.80%
Шымкент          12.93%     11.75%
Суммы долей по месяцам: [1. 1.]
Сумма равна 1 в сентябре: True
Сумма равна 1 в октябре: True
Доля Шымкента снизилась с 12.93% до 11.75%, изменение составило -1.19 п. п.


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [8]:
# 5
city_change = table[:, 1] - table[:, 0]
total_change = city_change.sum()
contribution = city_change / total_change * 100

print(f"{'Город':<12} {'Изменение, ₸':>14} {'Вклад, %':>10}")
for current_city, delta, part in zip(cities, city_change, contribution):
    print(f"{current_city:<12} {delta:>14,.0f} {part:>10.2f}")

shymkent_contribution = contribution[cities == "Шымкент"][0]
print(f"Начните проверку с Шымкента: на него приходится {shymkent_contribution:.2f}% общего падения выручки.")

Город          Изменение, ₸   Вклад, %
Актобе              -36,540       4.44
Алматы              141,950     -17.23
Астана             -183,690      22.30
Атырау               45,870      -5.57
Караганда          -276,840      33.60
Шымкент            -514,640      62.46
Начните проверку с Шымкента: на него приходится 62.46% общего падения выручки.


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [9]:
# 6
# table имеет форму (6, 2), а table.sum(axis=1) — (6,).
# При broadcasting размеры сравниваются справа, и 2 не совпадает с 6.
# keepdims=True сохраняет форму сумм (6, 1), как в примере из урока.
share = table / table.sum(axis=1, keepdims=True)

print(f"{'Город':<12} {'Сентябрь':>10} {'Октябрь':>10} {'Сумма равна 1':>15}")
for current_city, shares in zip(cities, share):
    row_sum = shares.sum()
    print(f"{current_city:<12} {shares[0]:>10.2%} {shares[1]:>10.2%} {str(round(row_sum, 6) == 1):>15}")

print("В каждом городе доли сентября и октября в выручке за два месяца в сумме дают 100%.")

Город          Сентябрь    Октябрь   Сумма равна 1
Актобе           50.36%     49.64%            True
Алматы           49.76%     50.24%            True
Астана           50.51%     49.49%            True
Атырау           48.79%     51.21%            True
Караганда        51.97%     48.03%            True
Шымкент          52.99%     47.01%            True
В каждом городе доли сентября и октября в выручке за два месяца в сумме дают 100%.


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [10]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [11]:
# 8